# Correctness and Invariants


```{contents}
:local:
:depth: 2
```

```{index} Make the Claim Strong Enough to Guide the Program
```



```{index} Make the Claim Strong Enough to Guide the Program
```

## Make the Claim Strong Enough to Guide the Program



A maximum routine is supposed to return the greatest value in a nonempty integer array. The phrase finds a maximum is too vague to determine empty behavior, duplicate handling or whether input may change. State the domain, output relation and mutation policy first. A proof cannot repair an unspecified contract after implementation choices have already been made.

Consider a left-to-right scan. After initializing best from the first element, let i denote the next unprocessed position. The invariant is that best is the maximum of positions zero through i-1. This relation mentions both the accumulated result and the processed boundary. Saying best is the maximum is not a useful invariant unless it explains maximum of which data at that point.

| Proof obligation | Reason for the scan |
| --- | --- |
| Initialization | The first element is the maximum of the one-element prefix |
| Maintenance | Comparing the next element extends the correct prefix by one |
| Exit | The processed prefix is the complete nonempty array |
| Termination | The number of unprocessed positions strictly decreases |

Initialization must match the domain. Setting best to zero fails on an all-negative nonempty input because zero may not occur in the array and is greater than every legitimate result. A positive-only test set would not expose this defect. The invariant makes the issue visible before a long list of tests is attempted.

For related explanation, see the further reading below.[^development-reading]


```{index} Partial Correctness and Termination Are Separate Questions
```



```{index} Partial Correctness and Termination Are Separate Questions
```

## Partial Correctness and Termination Are Separate Questions



An invariant helps establish what is true if execution reaches the exit. A decreasing nonnegative measure, such as remaining positions, explains why the loop eventually reaches that exit. These are different obligations. A loop can preserve a useful relation forever while failing to advance, so preservation alone does not establish total correctness.

Early returns need a local justification. In first-match search, returning at position i is correct only if that element matches and no earlier position matched. The scan's processed-prefix relation supplies the second fact. A return from the middle of an algorithm does not automatically invalidate proof reasoning, but it needs an argument at that exit.

Tests and counterexamples complement these arguments. Empty input checks the domain policy; one element checks initialization; all-negative values challenge a careless initial value; repeated maxima check that equality is handled consistently. A passing finite set supports confidence without proving the invariant for all accepted inputs. A single valid-domain counterexample, however, disproves a universal correctness claim.

Guided proof repair: assume best starts at zero for input -8, -3, -5. Identify exactly which initialization claim is false. Replace the initialization and trace the maintenance steps. Then state what the routine should do on empty input under two possible contracts: reject it explicitly or return a success signal with no maximum. Do not silently treat zero as the maximum of no values.


A fast expense report is useful only if its totals and selected records satisfy the business rules. Specify the result first, then connect every loop step to that specification.

```{index} Specifications and Failure Policies
```

## Specifications and Failure Policies




```{index} Specifications and Failure Policies
```


For a maximum, require a non-null, nonempty array. Promise that the result is an input value and is at least every input value. Merely promising a large value is insufficient: returning int.MaxValue would satisfy that weaker promise. Define rejected inputs and whether failure changes state.

In [ ]:
using System;
using System.Linq;

foreach (int[] values in new[] { new[] { -9, -2, -7 }, new[] { 5 }, Array.Empty<int>() })
{
    try { Console.WriteLine($"maximum={MaxExpense(values)}"); }
    catch (ArgumentException) { Console.WriteLine("empty input rejected"); }
}
long MaxExpense(int[] values)
{
    if (values is null) throw new ArgumentNullException(nameof(values));
    if (values.Length == 0) throw new ArgumentException("At least one expense is required.");
    int best = values[0];
    for (int i = 1; i < values.Length; i++)
        if (values[i] > best) best = values[i];
    return best;
}

maximum=-2
maximum=5
empty input rejected


Runtime guards enforce this contract. Other algorithms may document a precondition rather than check it each time. Binary search requires sorted input; checking that condition costs Θ(n) and changes the cost of an individual search.

**Common mistake.** Promising only that the result is "at least every input value". `int.MaxValue` satisfies that promise for any input, so the specification must also require that the result be one of the inputs; the cell's contract states both.


```{index} An Invariant with an Exact Boundary
```

## An Invariant with an Exact Boundary




```{index} An Invariant with an Exact Boundary
```


At the head of the maximum loop, `1 ≤ i ≤ n` and `best` is the maximum of the processed prefix `[0, i)`. Half-open notation includes index zero and excludes index `i`; it avoids uncertainty about whether the next value has already been processed.

In [ ]:
using System;
using System.Linq;

int[] values = { -9, -2, -7, -2 };
int best = values[0];
for (int i = 1; i < values.Length; i++)
{
    Console.WriteLine($"before i={i}: processed={i}, best={best}");
    if (values[i] > best) best = values[i];
    Console.WriteLine($"after i={i}: processed={i+1}, best={best}");
}
Console.WriteLine($"maximum={best}");

before i=1: processed=1, best=-9
after i=1: processed=2, best=-2
before i=2: processed=2, best=-2
after i=2: processed=3, best=-2
before i=3: processed=3, best=-2
after i=3: processed=4, best=-2
maximum=-2


Initialization: at `i=1`, the prefix contains only `values[0]`. Maintenance: comparing the next value extends the prefix by one while preserving its maximum. At normal loop exit `i=n`, the prefix is the entire input, so the invariant gives both parts of the postcondition.

**Common mistake.** Stating the invariant as "best is the maximum so far" without saying which indexes count. With a closed range `[0, i]` the claim is false at the loop head, because `values[i]` has not been compared yet; the half-open `[0, i)` is exact.


```{index} Code Completion: Assert the Invariant
```

### Code Completion: Assert the Invariant

A running-balance loop processes deposits and withdrawals and must never let the balance fall below zero. The invariant at the loop head is: `balance` equals the opening balance plus the sum of the processed prefix `[0, i)`, and that balance is nonnegative. Complete the two assertions so that the loop checks the invariant on every iteration, using the `check` helper, which throws when its condition is false. The expected output is `balance after 4 transactions = 70` followed by `invariant violated at i=1: balance would be -40` for the second transaction list.

In [ ]:
using System;

Run(new[] { 50, -30, 40, 10 }, opening: 0);
Run(new[] { 20, -60, 10 }, opening: 0);

static void Run(int[] transactions, int opening)
{
    long balance = opening, prefixSum = 0;
    try
    {
        for (int i = 0; i < transactions.Length; i++)
        {
            // Your code starts here.
            // 1. check that balance equals opening plus the sum of transactions[0..i)
            // 2. check that balance is nonnegative

            // Your code ends here.
            if (balance + transactions[i] < 0) throw new InvalidOperationException($"invariant violated at i={i}: balance would be {balance + transactions[i]}");
            balance += transactions[i];
            prefixSum += transactions[i];
        }
        Console.WriteLine($"balance after {transactions.Length} transactions = {balance}");
    }
    catch (InvalidOperationException ex) { Console.WriteLine(ex.Message); }
}

static void check(bool condition, string what)
{
    if (!condition) throw new InvalidOperationException("invariant check failed: " + what);
}

In [ ]:
// Solution
using System;

Run(new[] { 50, -30, 40, 10 }, opening: 0);
Run(new[] { 20, -60, 10 }, opening: 0);

static void Run(int[] transactions, int opening)
{
    long balance = opening, prefixSum = 0;
    try
    {
        for (int i = 0; i < transactions.Length; i++)
        {
            check(balance == opening + prefixSum, "balance tracks the processed prefix");
            check(balance >= 0, "balance is nonnegative");
            if (balance + transactions[i] < 0) throw new InvalidOperationException($"invariant violated at i={i}: balance would be {balance + transactions[i]}");
            balance += transactions[i];
            prefixSum += transactions[i];
        }
        Console.WriteLine($"balance after {transactions.Length} transactions = {balance}");
    }
    catch (InvalidOperationException ex) { Console.WriteLine(ex.Message); }
}

static void check(bool condition, string what)
{
    if (!condition) throw new InvalidOperationException("invariant check failed: " + what);
}

balance after 4 transactions = 70
invariant violated at i=1: balance would be -40


The checks never fire on either list, because the loop maintains the invariant; the rejection of the second list comes from the guard that refuses a withdrawal the balance cannot cover, which is the maintenance step protecting the nonnegativity half of the invariant. Assertions like these are how a proof becomes executable: they do not prove the loop correct for all inputs, but they turn a violated invariant into an immediate, located failure instead of a wrong total discovered weeks later. Production code usually keeps such checks behind a debug flag or in tests, since each costs time on every iteration.

```{index} Termination and Total Correctness
```

## Termination and Total Correctness




```{index} Termination and Total Correctness
```


An invariant explains the result if execution finishes: partial correctness. Total correctness also requires termination. A variant is a nonnegative integer measure that strictly decreases on each continuing iteration. For the maximum loop, `n−i` decreases by one.

In [ ]:
using System;
using System.Linq;

int remaining = 3;
while (remaining > 0)
{
    Console.WriteLine($"variant before={remaining}");
    remaining--;
}
Console.WriteLine($"variant at exit={remaining}");

variant before=3
variant before=2
variant before=1
variant at exit=0


An unchanged `remaining` could preserve a safety invariant while looping forever. The decreasing measure rules out that behavior. The third step of an invariant argument uses the exit condition to derive the result; it does not by itself prove that the exit will eventually occur.

**Common mistake.** Treating a true invariant as a proof that the loop finishes. `while (remaining > 0)` with no decrement keeps the invariant forever; the variant, a quantity that strictly decreases, is the separate argument that rules out running forever.


```{index} Three Proof Shapes
```

### Three Proof Shapes

The invariant argument on this page is one instance of a general technique, and it helps to know the three shapes that correctness arguments take.

| Shape | How it proceeds | Where it appears in this book |
| --- | --- | --- |
| Direct proof | Start from the preconditions and reason step by step to the postcondition | The early-return argument: the match at i plus "no earlier match" gives first-match |
| Proof by contradiction | Assume the claim is false and derive an impossibility | Any-match binary search: if the target were in a discarded half, the discarding comparison would have been wrong |
| Induction | Prove a base case, then show that truth for one step implies truth for the next | Every loop invariant: initialization is the base case, maintenance is the inductive step |

Induction deserves a second look because students often meet it as a mathematical curiosity and do not recognize it in code. A loop invariant proof *is* an induction on the number of iterations completed: the invariant holds after zero iterations by initialization, and if it holds after k iterations then maintenance shows it holds after k + 1. A recursive method's correctness is the same induction on the size of the input: the base case handles the smallest size directly, and the recursive case assumes the smaller call is correct, which is exactly the hypothesis induction grants. Chapter 17's `Count` and Chapter 22's merge sort are proved this way. The closed forms of Chapter 21.3's recurrences are verified the same way: guess the formula, check the base case, and show that if it holds for smaller sizes it holds for n.

```{index} Early Return and First-Match Search
```

## Early Return and First-Match Search




```{index} Early Return and First-Match Search
```


Specify first-match search: return the smallest matching index, or −1 when no match exists. Before testing index `i`, every index in `[0,i)` fails to match. Normal loop exit and early return require separate arguments.

In [ ]:
using System;
using System.Linq;

int[] values = { 5, 8, 8, 14 };
foreach (int target in new[] { 8, 14, 99 })
    Console.WriteLine($"target={target}, first index={FirstIndex(values,target)}");
int FirstIndex(int[] values, int target)
{
    for (int i = 0; i < values.Length; i++)
        if (values[i] == target) return i;
    return -1;
}

target=8, first index=1
target=14, first index=3
target=99, first index=-1


An early return identifies a matching value and the invariant excludes earlier matches. At normal exit, the invariant excludes every index. Empty input returns −1 without an array access. The variant `n−i` proves the continuing loop terminates.

**Common mistake.** Arguing only about the normal exit. The early `return i` leaves the loop before the exit condition holds, so its correctness needs its own argument: the match at `i` plus the invariant that no earlier index matched.


```{index} Boundary Tests and a Counterexample
```

## Boundary Tests and a Counterexample




```{index} Boundary Tests and a Counterexample
```


Tests complement proofs by challenging contracts and implementation details. A common maximum bug initializes best to zero; the bug stays hidden on positive data. Test singleton, negative, duplicate, extreme, empty, and rejected inputs.

In [ ]:
using System;
using System.Linq;

int[][] cases = { new[] { 0 }, new[] { -5,-2,-9 }, new[] { 7,7 }, new[] { int.MinValue,int.MaxValue } };
foreach (int[] values in cases)
{
    long actual = MaxExpense(values);
    if (actual != values.Max()) throw new Exception("Maximum mismatch.");
    Console.WriteLine($"[{string.Join(",",values)}] -> {actual}");
}
int broken = 0;
foreach (int value in new[] { -5,-2,-9 }) if (value > broken) broken = value;
Console.WriteLine($"zero-initialization counterexample: got={broken}, expected=-2");
long MaxExpense(int[] values)
{
    if (values is null) throw new ArgumentNullException(nameof(values));
    if (values.Length == 0) throw new ArgumentException("At least one expense is required.");
    int best = values[0];
    for (int i = 1; i < values.Length; i++)
        if (values[i] > best) best = values[i];
    return best;
}

[0] -> 0
[-5,-2,-9] -> -2
[7,7] -> 7
[-2147483648,2147483647] -> 2147483647
zero-initialization counterexample: got=0, expected=-2


The reference comparison checks actual code on these inputs. The invariant covers every input satisfying the contract. Arithmetic behavior is part of the specification too: use a sufficiently wide type or a checked operation when a mathematical sum may overflow the chosen C# type.

**Common mistake.** Initializing `best` to 0. Every positive test passes, and the all-negative input returns 0, a value that is not in the array; initialize from `values[0]` after the nonempty check, as `MaxExpense` does.


```{index} Code Tracing: Fill the Invariant Table
```

### Code Tracing: Fill the Invariant Table

The loop below computes the *second-largest* distinct value of a nonempty array, or reports that there is none. At the head of each iteration write `i`, `best`, `second`, and whether the stated invariant holds: `best` is the maximum of `[0, i)` and `second` is the largest value in `[0, i)` strictly less than `best`, or `null` if no such value exists. Then predict the output for both arrays. Is the invariant strong enough to prove the postcondition at exit?

```csharp
int? best = null, second = null;
for (int i = 0; i < values.Length; i++)
{
    int v = values[i];
    if (best is null || v > best) { second = best; best = v; }
    else if (v < best && (second is null || v > second)) second = v;
}
```

In [ ]:
// Solution
using System;

foreach (int[] values in new[] { new[] { 40, 10, 40, 25, 30 }, new[] { 7, 7, 7 } })
{
    int? best = null, second = null;
    Console.WriteLine($"input [{string.Join(", ", values)}]");
    for (int i = 0; i < values.Length; i++)
    {
        Console.WriteLine($"  head i={i}: best={best?.ToString() ?? "null"}, second={second?.ToString() ?? "null"}");
        int v = values[i];
        if (best is null || v > best) { second = best; best = v; }
        else if (v < best && (second is null || v > second)) second = v;
    }
    Console.WriteLine($"  result: {(second is null ? "no second-largest distinct value" : $"second largest = {second}")}");
}

input [40, 10, 40, 25, 30]
  head i=0: best=null, second=null
  head i=1: best=40, second=null
  head i=2: best=40, second=10
  head i=3: best=40, second=10
  head i=4: best=40, second=25
  result: second largest = 30
input [7, 7, 7]
  head i=0: best=null, second=null
  head i=1: best=7, second=null
  head i=2: best=7, second=null
  result: no second-largest distinct value


For the first array the table runs best null, 40, 40, 40, 40 and second null, null, 10, 10, 25, and the result is 30. The duplicate 40 at index 2 changes nothing, because the first branch requires strictly greater and the second requires strictly less; that is what keeps `second` distinct from `best`. For the all-7 array `second` stays `null` and the loop reports that no second-largest distinct value exists, which the invariant predicts: no value in the prefix is ever strictly less than `best`. The invariant is strong enough because at exit the prefix is the whole array, and the two clauses are exactly the postcondition's two clauses; an invariant that tracked only `best` would leave `second` unproved.

```{index} Check Your Understanding
```

### Check Your Understanding

Answer each question before opening its dropdown.

1. A loop passes 10,000 random tests. What does that establish, and what does it not?

```{dropdown} Answer
It establishes that the loop is correct on those 10,000 inputs and gives some confidence about similar inputs. It does not establish correctness for all inputs in the domain, which only an invariant argument covers, and random tests rarely hit the boundaries, empty, singleton, all-equal, or extreme values, where defects concentrate.
```

2. Why is "best is the largest value seen so far" a weaker invariant than "best is the maximum of values[0..i)"?

```{dropdown} Answer
The first does not say which values have been seen, so it cannot be checked at a specific moment and cannot be combined with the exit condition to conclude anything about the whole array. The half-open range ties the claim to the loop variable, which is what makes the exit step work.
```

3. A recursive method is proved correct by assuming its recursive call returns the right answer. Is that circular?

```{dropdown} Answer
No; it is induction on input size. The assumption is made only about strictly smaller inputs, and the base case anchors the smallest size directly. The argument is circular only if the recursive call is not on a smaller input, which is also exactly when the method fails to terminate.
```

4. Which proof shape fits this claim: "no comparison sort can sort every input of n distinct keys with fewer than about n log n comparisons"?

```{dropdown} Answer
Contradiction, combined with counting. Assume an algorithm used fewer comparisons; then it has too few distinct outcomes to distinguish all n! input orders, so two different orders would receive the same output and one of them would be sorted wrongly. Chapter 23 develops this lower bound.
```

```{index} Exercise
```

## Exercise




```{index} Analysis practice
```

Count nonnegative adjustments. State the processed-prefix invariant and variant; include zero as nonnegative.

In [ ]:
using System;

// Exercise
// Count nonnegative adjustments. State the processed-prefix invariant and variant; include zero as nonnegative.
// Your code starts here.
/* TODO */
// Your code ends here.

In [1]:
using System;

// Solution
int[] values={-3,0,5,-1};
int count=0;
// At loop head, count is the number of nonnegative entries in [0,i).
// Variant is values.Length-i.
for(int i=0;i<values.Length;i++)if(values[i]>=0)count++;
Console.WriteLine($"nonnegative adjustments={count}");

nonnegative adjustments=2


The output is `nonnegative adjustments=2`, counting the 0 and the 5. Zero passes the `>=` test, which the specification required; a test with `> 0` would have counted one and been wrong in a way no exception reveals.


```{rubric} Footnotes
```


[^2102_correctness-1]: A specification must identify the allowed inputs, promised result, and failure behavior; a proof is only as useful as that specification.
[^2102_correctness-2]: Testing a finite sample generally cannot prove a statement over all possible input sizes. Boundary tests and counterexamples remain valuable implementation checks.

[^development-reading]: Further reading: [OpenDSA CS3, Common Misunderstandings](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/AnalMisunderstanding.html). The worked business scenarios and guided reasoning here are original teaching examples.
